In [1]:
!pip install langchain chromadb huggingface tiktoken pypdf langchain openai langchain community 

In [2]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

In [3]:
from langchain_core.documents import Document

doc1 = Document(
    page_content="Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.",
    metadata={"team": "Royal Challengers Bangalore"}
)

doc2 = Document(
    page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
    metadata={"team": "Mumbai Indians"}
)

doc3 = Document(
    page_content="MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.",
    metadata={"team": "Chennai Super Kings"}
)

doc4 = Document(
    page_content="Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.",
    metadata={"team": "Mumbai Indians"}
)

doc5 = Document(
    page_content="Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.",
    metadata={"team": "Chennai Super Kings"}
)


docs = [doc1, doc2, doc3, doc4, doc5]

In [4]:
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

vectorstore = Chroma(
    collection_name="IPL-Players",
    embedding_function=embedding_model,
    persist_directory="ChromaDB-IPL"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [5]:
vectorstore.add_documents(docs)
print("Documents added!")

Documents added!


In [6]:
vectorstore.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['851fa99b-da96-4186-9e7c-7eb522607e36',
  'de8014a5-20af-4f9a-986d-b31e07b01c28',
  '3c650b78-5d0e-40f2-b8be-f840321f2da9',
  '6a9aab92-018a-4bdd-a910-27cbf2850b1a',
  '8b5f9326-be3b-456d-8ace-e02d52795456'],
 'embeddings': array([[ 0.00994726,  0.06914335, -0.05147116, ..., -0.03543343,
          0.01284809,  0.01248291],
        [ 0.00127747,  0.03129849, -0.02375384, ..., -0.00518364,
         -0.03280614,  0.02737715],
        [-0.10265912,  0.0265081 ,  0.02271502, ..., -0.03359751,
         -0.07984938, -0.0150771 ],
        [ 0.02123393, -0.02468546, -0.0449437 , ..., -0.10995812,
          0.00572556,  0.09915379],
        [ 0.01873976,  0.04382845, -0.04304254, ..., -0.07801623,
         -0.07840687, -0.0030419 ]], shape=(5, 384)),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the mo

In [7]:
vectorstore.similarity_search(
    query='Who among these are a bowler?',
    k=2
)

[Document(id='6a9aab92-018a-4bdd-a910-27cbf2850b1a', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
 Document(id='de8014a5-20af-4f9a-986d-b31e07b01c28', metadata={'team': 'Mumbai Indians'}, page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.")]

In [8]:
# search with similarity score
vectorstore.similarity_search_with_score(
    query='Who among these are a bowler?',
    k=2
)

[(Document(id='6a9aab92-018a-4bdd-a910-27cbf2850b1a', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
  0.9693599939346313),
 (Document(id='de8014a5-20af-4f9a-986d-b31e07b01c28', metadata={'team': 'Mumbai Indians'}, page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure."),
  1.149344801902771)]

In [9]:
# meta-data filtering
vectorstore.similarity_search_with_score(
    query="",
    filter={"team": "Chennai Super Kings"}
)

[(Document(id='3c650b78-5d0e-40f2-b8be-f840321f2da9', metadata={'team': 'Chennai Super Kings'}, page_content='MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.'),
  1.8436006307601929),
 (Document(id='8b5f9326-be3b-456d-8ace-e02d52795456', metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
  1.8909375667572021)]

In [10]:
# update documents
updated_doc1 = Document(
    page_content="Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple centuries in a single season. Despite RCB not winning an IPL title under his captaincy, Kohli's passion and fitness set a benchmark for the league. His ability to chase targets and anchor innings has made him one of the most dependable players in T20 cricket.",
    metadata={"team": "Royal Challengers Bangalore"}
)

vectorstore.update_document(document_id='09a39dc6-3ba6-4ea7-927e-fdda591da5e4', document=updated_doc1)

In [11]:
# view documents
vectorstore.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['851fa99b-da96-4186-9e7c-7eb522607e36',
  'de8014a5-20af-4f9a-986d-b31e07b01c28',
  '3c650b78-5d0e-40f2-b8be-f840321f2da9',
  '6a9aab92-018a-4bdd-a910-27cbf2850b1a',
  '8b5f9326-be3b-456d-8ace-e02d52795456'],
 'embeddings': array([[ 0.00994726,  0.06914335, -0.05147116, ..., -0.03543343,
          0.01284809,  0.01248291],
        [ 0.00127747,  0.03129849, -0.02375384, ..., -0.00518364,
         -0.03280614,  0.02737715],
        [-0.10265912,  0.0265081 ,  0.02271502, ..., -0.03359751,
         -0.07984938, -0.0150771 ],
        [ 0.02123393, -0.02468546, -0.0449437 , ..., -0.10995812,
          0.00572556,  0.09915379],
        [ 0.01873976,  0.04382845, -0.04304254, ..., -0.07801623,
         -0.07840687, -0.0030419 ]], shape=(5, 384)),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the mo

In [12]:
# delete document
vectorstore.delete(ids=['09a39dc6-3ba6-4ea7-927e-fdda591da5e4'])